In [1]:
import pandas as pd
import scanpy as sc
import plotnine as p9

import liana as li
import cell2cell as c2c
import decoupler as dc # needed for pathway enrichment

import warnings
warnings.filterwarnings('ignore')
from collections import defaultdict

%matplotlib inline

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [2]:
use_gpu = False

## Load & Prep Data

In [3]:
adata = sc.read_h5ad('/Users/annamaguza/Desktop/data/gut_data/gut_hs_fetal_AM_06032025_142621_raw_velocity.h5ad')
adata

AnnData object with n_obs × n_vars = 317530 × 43704
    obs: 'cell_id', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'consensus_fraction', 'consensus_passed_qc', 'total_counts', 'n_genes_by_counts', 'percent_chrY', 'XIST-counts', 'XIST-percentage', 'sex', 'S_score', 'G2M_score', 'Cell_cycle_phase', 'Study_name', 'ArrayExpress_ID', 'library_preparation_protoc

In [4]:
del adata.layers

In [5]:
sc.pp.normalize_total(adata, target_sum=1e6, exclude_highly_expressed=True)
sc.pp.log1p(adata)

## Ligand-Receptor Inference by Sample

In [6]:
adata

AnnData object with n_obs × n_vars = 317530 × 43704
    obs: 'cell_id', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'consensus_fraction', 'consensus_passed_qc', 'total_counts', 'n_genes_by_counts', 'percent_chrY', 'XIST-counts', 'XIST-percentage', 'sex', 'S_score', 'G2M_score', 'Cell_cycle_phase', 'Study_name', 'ArrayExpress_ID', 'library_preparation_protoc

In [7]:
adata.obs['cell_states'] = adata.obs['cell_states'].astype('str') + '_' + adata.obs['leiden_cluster'].astype('str')
adata.obs['cell_states'] = adata.obs['cell_states'].str.replace(r'_nan', '', regex=True)
adata.obs['cell_states'] = adata.obs['cell_states'].astype('category')

In [8]:
adata = adata[adata.obs['sample_id'] != 'FCA_gut8015060']

In [9]:
li.mt.rank_aggregate.by_sample(
    adata,
    groupby='cell_states',
    resource_name='consensus', 
    sample_key='sample_id', 
    use_raw=False,
    verbose=True, 
    n_perms=None, 
    return_all_lrs=True,
    )

Now running: Human_colon_16S8159190: 100%|███| 56/56 [2:11:34<00:00, 140.97s/it]


In [10]:
adata.uns["liana_res"].sort_values("magnitude_rank").head(10)

,sample_id,source,target,ligand_complex,receptor_complex,lr_means,expr_prod,scaled_weight,lr_logfc,spec_weight,lrscore,magnitude_rank
129906041,HT-228-duodenum,Classical monocytes,ILCP,S100A9,ITGB2,9.138235,81.285355,6.900373,8.682291,0.020101,0.972699,6.389631e-14
75599162,HT-158-duodenum-Unsorted,IgM plasma cell,ILC2,LGALS1,CD69,8.444268,69.966652,2.391335,2.945939,0.001967,0.972588,7.158520e-14
57595370,HT-158-duodenum-Sorted,Stromal 3 (KCNN3+),Activated CD8 T,LGALS1,CD69,8.641468,74.337761,2.507844,3.326234,0.002197,0.974834,1.072073e-13
97579918,HT-172-duodenum-1,mature Goblet cell,PreB cells,TFF3,CXCR4,9.358441,84.252106,5.286127,6.869717,0.013775,0.970176,1.262776e-13
224211478,Human_colon_16S7985389,L cells (PYY+),Venous EC,GCG,RAMP2,9.104130,82.022293,9.310904,7.423591,0.063637,0.971762,1.421856e-13
253682590,Human_colon_16S7985391,K cells (GIP+),cycling EC,GIP,RAMP2,9.779734,91.813156,7.019156,9.450136,0.036903,0.965770,1.594357e-13
239734678,Human_colon_16S7985390,mature Goblet cell,PreB cells,TFF3,CXCR4,9.420482,85.523552,2.722587,5.435760,0.005089,0.965545,1.695432e-13
267975566,Human_colon_16S7985392,mature Goblet cell,pDC,TFF3,CXCR4,9.186930,82.199875,2.385902,4.612250,0.002376,0.962866,1.857180e-13
153056777,HT-234-duodenum,Glia,Macrophages,APP,CD74,8.586891,70.059303,1.912265,3.436738,0.001688,0.955165,2.016913e-13
129906042,HT-228-duodenum,Classical monocytes,ILCP,S100A8,ITGB2,9.019518,79.469543,14.759940,9.513805,0.031506,0.972397,2.555852e-13


In [11]:
df = adata.uns["liana_res"].sort_values("magnitude_rank")

In [12]:
# leave only if source or target is a stem cell
df = df[(df['source'].str.contains('Stem')) | (df['target'].str.contains('Stem'))]

In [13]:
df

,sample_id,source,target,ligand_complex,receptor_complex,lr_means,expr_prod,scaled_weight,lr_logfc,spec_weight,lrscore,magnitude_rank
10748013,BRC2026_IL_total_cells,Arterial EC,Stem cells_0,PRND,RPSA,7.639039,57.674507,3.724692,4.529953,0.024292,0.943790,9.130952e-12
3369747,BRC2026_DU_total_cells,Arterial EC,Stem cells_1,PRND,RPSA,7.413902,53.097298,3.629122,4.400154,0.012334,0.954126,3.772565e-11
292200741,Human_colon_16S7985395,Arterial EC,Stem cells_0,PRND,RPSA,7.517251,55.370518,3.733154,4.814643,0.018950,0.954167,5.978157e-11
48093049,BRC2133_DU_pos_cells,Cycling pericyte,Stem cells_0,PTN,NCL,7.439415,54.359821,1.590940,2.168014,0.002164,0.950819,7.003307e-11
3369748,BRC2026_DU_total_cells,Arterial EC,Stem cells_0,PRND,RPSA,7.338116,52.180756,3.527572,4.280949,0.012122,0.953743,7.132502e-11
...,...,...,...,...,...,...,...,...,...,...,...,...
102058386,HT-172-duodenum-1,D cells (SST+),Stem cells_0,BMP2,ACVR2A_BMPR1A,0.482423,0.232677,-1.400065,-5.573085,0.000006,0.630932,1.000000e+00
102058392,HT-172-duodenum-1,D cells (SST+),Stem cells_0,BMP2,GPC1,0.482423,0.232677,-1.400065,-5.573085,0.000006,0.630932,1.000000e+00
102057626,HT-172-duodenum-1,D cells (SST+),Stem cells_0,COL26A1,ITGA10_ITGB1,0.482423,0.232677,-1.400065,-5.573085,0.000006,0.630932,1.000000e+00
102057625,HT-172-duodenum-1,D cells (SST+),Stem cells_0,COL24A1,ITGA2_ITGB1,0.482423,0.232677,-1.400065,-5.573085,0.000006,0.630932,1.000000e+00


In [14]:
df = df.head(1000)

In [15]:
df.to_csv('/Users/annamaguza/Desktop/data/gut_data/liana_fetal_by_sample_top1000_stem_involved.csv')

## Building a Tensor

In [16]:
tensor = li.multi.to_tensor_c2c(adata,
                                sample_key='sample_id',
                                score_key='magnitude_rank', # can be any score from liana
                                how='outer_cells' # how to join the samples
                                )

100%|███████████████████████████████████████████| 56/56 [03:59<00:00,  4.28s/it]


In [17]:
tensor.tensor.shape

(56, 23, 97, 97)

In [18]:
c2c.io.export_variable_with_pickle(tensor, "/Users/annamaguza/Desktop/data/gut_data/liana_fetal_gut_tensor.pkl")

/Users/annamaguza/Desktop/data/gut_data/liana_fetal_gut_tensor.pkl  was correctly saved.


In [19]:
adata.obs

,cell_id,Source Name,ENA_SAMPLE,BioSD_SAMPLE,organism,disease,organism_part,cell_type,growth_condition,developmental_stage,...,batch,sampling_site,cell_index,celltype,library_construnction_and_layout,cell_states,cellstates_scANVI,confidence_score,gut_region,leiden_cluster
cell_index,,,,,,,,,,,,,,,,,,,,,
AAACCTGCACACCGAC-1-4918STDY7717789_ERR4010660,AAACCTGCACACCGAC-1-4918STDY7717789,BRC2133_CO_neg_cells,ERS4414926,SAMEA6655457,Homo sapiens,normal,colon,,primary tissue,9th week post-fertilization human stage,...,4918STDY7717789,NaN,AAACCTGCACACCGAC-1-4918STDY7717789_ERR4010660,Mesenchymal,10xV2_PAIRED,SMC (PLPP2+),SMC,0.998176,large intestine,NaN
AAACCTGGTCAAAGCG-1-4918STDY7717789_ERR4010660,AAACCTGGTCAAAGCG-1-4918STDY7717789,BRC2133_CO_neg_cells,ERS4414926,SAMEA6655457,Homo sapiens,normal,colon,,primary tissue,9th week post-fertilization human stage,...,4918STDY7717789,NaN,AAACCTGGTCAAAGCG-1-4918STDY7717789_ERR4010660,Mesenchymal,10xV2_PAIRED,Mesoderm 2(ZEB2+),Mesoderm,0.684551,large intestine,NaN
AAACGGGAGGCTATCT-1-4918STDY7717789_ERR4010660,AAACGGGAGGCTATCT-1-4918STDY7717789,BRC2133_CO_neg_cells,ERS4414926,SAMEA6655457,Homo sapiens,normal,colon,,primary tissue,9th week post-fertilization human stage,...,4918STDY7717789,NaN,AAACGGGAGGCTATCT-1-4918STDY7717789_ERR4010660,Mesenchymal,10xV2_PAIRED,SMC (PLPP2+),SMC,1.000000,large intestine,NaN
AAACGGGAGTCTCAAC-1-4918STDY7717789_ERR4010660,AAACGGGAGTCTCAAC-1-4918STDY7717789,BRC2133_CO_neg_cells,ERS4414926,SAMEA6655457,Homo sapiens,normal,colon,,primary tissue,9th week post-fertilization human stage,...,4918STDY7717789,NaN,AAACGGGAGTCTCAAC-1-4918STDY7717789_ERR4010660,Neuronal,10xV2_PAIRED,Progenitor,Progenitor,0.999998,large intestine,NaN
AAACGGGGTCGAACAG-1-4918STDY7717789_ERR4010660,AAACGGGGTCGAACAG-1-4918STDY7717789,BRC2133_CO_neg_cells,ERS4414926,SAMEA6655457,Homo sapiens,normal,colon,,primary tissue,9th week post-fertilization human stage,...,4918STDY7717789,NaN,AAACGGGGTCGAACAG-1-4918STDY7717789_ERR4010660,Mesenchymal,10xV2_PAIRED,SMC (PLPP2+),SMC,0.999985,large intestine,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTAGGCAGTCTCTCTG-1-Human_colon_16S8159189_ERR4574299,TTAGGCAGTCTCTCTG-1-Human_colon_16S8159189,Human_colon_16S8159189,ERS5054897,SAMEA7296331,Homo sapiens,normal,ileum,NaN,primary tissue,embryo,...,Human_colon_16S8159189,terminal,TTAGGCAGTCTCTCTG-1-Human_colon_16S8159189_ERR4...,Epithelial,10xV2_PAIRED,Stem cells_1,Stem cells,0.919425,small intestine,1
TTATGCTGTCTAAACC-1-Human_colon_16S8159189_ERR4574299,TTATGCTGTCTAAACC-1-Human_colon_16S8159189,Human_colon_16S8159189,ERS5054897,SAMEA7296331,Homo sapiens,normal,ileum,NaN,primary tissue,embryo,...,Human_colon_16S8159189,terminal,TTATGCTGTCTAAACC-1-Human_colon_16S8159189_ERR4...,Epithelial,10xV2_PAIRED,Enterocyte,Enterocyte,0.999809,small intestine,NaN
TTGAACGTCAAGGCTT-1-Human_colon_16S8159189_ERR4574299,TTGAACGTCAAGGCTT-1-Human_colon_16S8159189,Human_colon_16S8159189,ERS5054897,SAMEA7296331,Homo sapiens,normal,ileum,NaN,primary tissue,embryo,...,Human_colon_16S8159189,terminal,TTGAACGTCAAGGCTT-1-Human_colon_16S8159189_ERR4...,Epithelial,10xV2_PAIRED,BEST4+ epithelial,BEST4+ epithelial,0.990475,small intestine,NaN


In [20]:
context_dict = adata.obs[['sample_id', 'age_group']].drop_duplicates()
context_dict = dict(zip(context_dict['sample_id'], context_dict['age_group']))
context_dict = defaultdict(lambda: 'Unknown', context_dict)

tensor_meta = c2c.tensor.generate_tensor_metadata(interaction_tensor=tensor,
                                                  metadata_dicts=[context_dict, None, None, None],
                                                  fill_with_order_elements=True
                                                  )

In [22]:
tensor = c2c.analysis.run_tensor_cell2cell_pipeline(tensor,
                                                    tensor_meta,
                                                    copy_tensor=True, # Whether to output a new tensor or modifying the original
                                                    rank=None, # Number of factors to perform the factorization. If None, it is automatically determined by an elbow analysis. Here, it was precomuputed.
                                                    tf_optimization='regular', # To define how robust we want the analysis to be.
                                                    random_state=0, # Random seed for reproducibility
                                                    device='cpu', # Device to use. If using GPU and PyTorch, use 'cuda'. For CPU use 'cpu'
                                                    elbow_metric='error', # Metric to use in the elbow analysis.
                                                    smooth_elbow=False, # Whether smoothing the metric of the elbow analysis.
                                                    upper_rank=20, # Max number of factors to try in the elbow analysis
                                                    tf_init='random', # Initialization method of the tensor factorization
                                                    tf_svd='numpy_svd', # Type of SVD to use if the initialization is 'svd'
                                                    cmaps=None, # Color palettes to use in color each of the dimensions. Must be a list of palettes.
                                                    sample_col='Element', # Columns containing the elements in the tensor metadata
                                                    group_col='Category', # Columns containing the major groups in the tensor metadata
                                                    output_fig=False, # Whether to output the figures. If False, figures won't be saved a files if a folder was passed in output_folder.
                                                    )

Running Elbow Analysis


 45%|████████████████▋                    | 9/20 [8:54:14<10:52:57, 3561.56s/it]


KeyboardInterrupt: 

In [ ]:
factors, axes = c2c.plotting.tensor_factors_plot(interaction_tensor=tensor,
                                                 metadata = tensor_meta, # This is the metadata for each dimension
                                                 sample_col='Element',
                                                 group_col='Category',
                                                 meta_cmaps = ['magma_r', 'Dark2_r', 'tab20', 'tab20'],
                                                 fontsize=10, # Font size of the figures generated
                                                 )

## Factorization Results

In [ ]:
factors = tensor.factors

In [ ]:
factors.keys()

In [ ]:
_ = c2c.plotting.context_boxplot(context_loadings=factors['Contexts'],
                                 metadict=context_dict,
                                 nrows=2,
                                 figsize=(8, 6),
                                 statistical_test='t-test_ind',
                                 pval_correction='fdr_bh',
                                 cmap='plasma',
                                 verbose=False,
                                )

In [ ]:
c2c.plotting.ccc_networks_plot(factors,
                               included_factors=['Factor 6'],
                               network_layout='circular',
                               ccc_threshold=0.05, # Only important communication
                               nrows=1,
                               panel_size=(8, 8), # This changes the size of each figure panel.
                              )